In [ ]:
import torch
from transformers import pipeline

NLI_MODEL = "cross-encoder/nli-deberta-v3-small"

_verifier = None


def get_verifier():
    global _verifier

    if _verifier is None:
        device = 0 if torch.cuda.is_available() else -1

        _verifier = pipeline(
            "text-classification",
            model=NLI_MODEL,
            device=device
        )

    return _verifier


def verify_claim(claim, evidence, threshold=0.70):
    """
    Estimate whether the evidence entails, contradicts, or is neutral
    toward a claim. This is a model prediction, not a truth guarantee.
    """
    claim = claim.strip()
    evidence = evidence.strip()

    if not claim or not evidence:
        return {
            "label": "uncertain",
            "confidence": 0.0,
            "warning": "Claim or evidence is empty."
        }

    classifier = get_verifier()

    # NLI convention: premise = evidence, hypothesis = claim.
    result = classifier({
        "text": evidence[:4000],
        "text_pair": claim[:1000]
    })[0]

    raw_label = result["label"].lower()
    confidence = float(result["score"])

    if "entail" in raw_label and confidence >= threshold:
        label = "supported"
    elif "contrad" in raw_label and confidence >= threshold:
        label = "contradicted"
    else:
        label = "uncertain"

    return {
        "label": label,
        "model_label": raw_label,
        "confidence": confidence,
        "warning": (
            "This is an automated evidence check, not a guarantee "
            "of factual correctness."
        )
    }


def verify_answer(answer, sources, threshold=0.70):
    """
    Baseline whole-answer check against each retrieved passage.
    For a stronger evaluation, split answers into individual claims.
    """
    if not sources:
        return {
            "label": "uncertain",
            "confidence": 0.0,
            "checks": []
        }

    checks = []

    for source in sources:
        check = verify_claim(
            answer,
            source["text"],
            threshold=threshold
        )

        check["document"] = source["document"]
        check["page"] = source["page"]
        checks.append(check)

    if any(item["label"] == "supported" for item in checks):
        overall = "potentially supported"
    elif any(item["label"] == "contradicted" for item in checks):
        overall = "possible contradiction"
    else:
        overall = "uncertain"

    return {
        "label": overall,
        "confidence": max(
            (item["confidence"] for item in checks),
            default=0.0
        ),
        "checks": checks
    }
